# Clase 36 - Notebook 3 - Función de Deseabilidad Compuesta (DFA)

Este notebook implementa el enfoque de **Función de Deseabilidad** de Harrington / Derringer & Suich para resolver el problema biobjetivo.


### ⚙️ Paso 0: Configuración Automática y Persistencia en Google Drive
Si estás en **Google Colab**, ejecuta la siguiente celda. Se montará tu Google Drive y se clonará automáticamente el repositorio en tu carpeta `MyDrive/DAII-SprayDrying`, garantizando que todos tus cambios, figuras y archivos `.csv` de resultados queden **guardados permanentemente en tu cuenta de Google Drive**.


In [ ]:
# --- Preámbulo Universal con Persistencia en Google Drive ---
import os, sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    
    PROJECT_DIR = '/content/drive/MyDrive/DAII-SprayDrying'
    if not os.path.exists(PROJECT_DIR):
        print('Clonando repositorio en tu Google Drive...')
        !git clone https://github.com/felipehuerta17/DAII-SprayDrying.git {PROJECT_DIR}
    
    os.chdir(PROJECT_DIR)
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)
        
    %pip install -q numpy scipy pandas matplotlib pymoo casadi
    print(f'✅ Entorno listo. Modificaciones y resultados guardados en: {PROJECT_DIR}')
else:
    print('✅ Ejecutando en entorno local.')


## Ejecución de Optimización por Deseabilidad


In [ ]:
import os, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
from spraydrylib.desirability import dfa_front, Bounds

os.makedirs("./outputs", exist_ok=True)
bounds = Bounds(lb=np.array([462.0, 3.5e-5]), ub=np.array([858.0, 6.5e-5]))
Ls = (16.0, 0.045); Us = (30.0, 0.070)

# Muestreo de 21 combinaciones de pesos en el intervalo [0, 1]
w1 = np.linspace(0.05, 0.95, 21)
weights = [(float(wi), float(1.0 - wi)) for wi in w1]

t0 = time.perf_counter()
X, F, D = dfa_front(weights, bounds, Ls, Us, tf=400.0, n_steps=600, pop_size=20, iters=15, seed=7)
elapsed = time.perf_counter() - t0
print(f"Tiempo aprox: {elapsed:.2f} s")

stamp = time.strftime("%Y%m%d-%H%M%S")
csv = f'./outputs/front_dfa_{stamp}.csv'
pd.DataFrame(np.hstack([X,F]), columns=["G_kg_h","rd_m","Energia_kW","Xo_prom_ultimos"]).to_csv(csv, index=False)
print("Guardado:", csv, "| puntos:", len(X))

plt.figure(figsize=(7.5,4.5))
plt.scatter(F[:,0], F[:,1], s=28, facecolor="#ff7f0e", edgecolor="black", alpha=0.9, label="Frente de Pareto - DFA")
plt.xlabel("Gasto energético (kW)")
plt.ylabel(r"Contenido de agua en el droplet $\left(\frac{kg_{agua}}{kg_{sólidos}}\right)$")
plt.title("Frente por función de deseabilidad (DFA)")
plt.legend(); plt.tight_layout()
plt.savefig(f'./outputs/pareto_dfa_{stamp}.png', dpi=200); plt.savefig(f'./outputs/pareto_dfa_{stamp}.svg')
plt.show()
